# Task 2 - AlexNet vs DHVT on CIFAR-100 Variants

This notebook compares **AlexNet** and **DHVT** on a small controlled benchmark derived from **CIFAR-100**, with the goal of highlighting how the two models react to different visual conditions.

**Objective.** Build a smaller benchmark based on CIFAR-100 and compare the behaviour of AlexNet and DHVT under several data conditions, rather than only on the original clean dataset.

**Protocol.** A balanced subset of 12 CIFAR-100 classes is used. For each variant, the two models are trained and evaluated under the same settings. The four variants are:
- `clean`: baseline small balanced subset
- `texture`: local patch shuffling keeps texture cues but weakens object layout
- `long_range`: downsample-then-upsample transformation reduces fine local detail and emphasizes coarse global structure
- `damaged_occluded`: deterministic damage and occlusion hide part of the object

**Expected reading.** The benchmark is designed to show whether DHVT keeps its advantage over AlexNet not only on standard classification, but also when the image statistics are deliberately modified.

**Outputs.** The notebook saves per-variant training logs, a final comparison summary, and CSV files that can be reused directly in the report.


In [1]:
import csv
import random
from collections import defaultdict
from pathlib import Path

import math
import matplotlib.pyplot as plt
import numpy as np
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from PIL import Image
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets


In [2]:
runtime_mode = 'colab'  # 'local' or 'colab'

if runtime_mode == 'colab':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    project_root = Path('/content/drive/MyDrive/classification-deep-learning-project')
    data_dir = Path('/content/data')
else:
    project_root = Path.cwd()
    data_dir = project_root / 'data'

model_dir = project_root / 'models'
prediction_dir = project_root / 'predictions'
report_figure_dir = project_root / 'report' / 'figures'

for folder in [data_dir, model_dir, prediction_dir, report_figure_dir]:
    folder.mkdir(parents=True, exist_ok=True)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Runtime mode:', runtime_mode)
print('Device:', device)
print('Seed:', seed)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Mounted at /content/drive
Runtime mode: colab
Device: cuda
Seed: 42
GPU: Tesla T4


In [3]:
dataset_name = 'cifar100_v4_small'
experiment_variants = ['clean', 'texture', 'long_range', 'damaged_occluded']

cnn_image_size = 224
dhvt_image_size = 32
cnn_batch_size = 64
dhvt_batch_size = 64
num_epochs = 8
alexnet_learning_rate = 0.001
dhvt_learning_rate = 5e-4
weight_decay = 5e-4
num_workers = 0

selected_cifar100_classes = [
    'bridge', 'bus', 'castle', 'leopard',
    'orchid', 'poppy', 'road', 'rose',
    'skyscraper', 'tiger', 'train', 'willow_tree'
]
train_samples_per_class = 80
val_samples_per_class = 30

challenge_variant_descriptions = {
    'clean': 'baseline balanced subset without synthetic corruption',
    'texture': 'local patches are shuffled to keep texture cues but weaken global layout',
    'long_range': 'fine texture is suppressed so coarse global structure matters more',
    'damaged_occluded': 'a deterministic cutout and damage pattern hide part of the object',
}

print('Variants:', experiment_variants)
print('Epochs:', num_epochs)
print('Selected classes:', selected_cifar100_classes)


Variants: ['clean', 'texture', 'long_range', 'damaged_occluded']
Epochs: 8
Selected classes: ['bridge', 'bus', 'castle', 'leopard', 'orchid', 'poppy', 'road', 'rose', 'skyscraper', 'tiger', 'train', 'willow_tree']


In [4]:
alexnet_train_transform = transforms.Compose([
    transforms.Resize((cnn_image_size, cnn_image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

alexnet_test_transform = transforms.Compose([
    transforms.Resize((cnn_image_size, cnn_image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dhvt_train_transform = transforms.Compose([
    transforms.Resize((dhvt_image_size, dhvt_image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(dhvt_image_size, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dhvt_test_transform = transforms.Compose([
    transforms.Resize((dhvt_image_size, dhvt_image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


def build_balanced_subset_indices(targets, selected_original_labels, samples_per_class, rng):
    indices_by_label = defaultdict(list)
    for sample_index, label in enumerate(targets):
        if label in selected_original_labels:
            indices_by_label[label].append(sample_index)

    balanced_indices = []
    for original_label in selected_original_labels:
        class_indices = list(indices_by_label[original_label])
        rng.shuffle(class_indices)
        balanced_indices.extend(class_indices[:samples_per_class])

    return balanced_indices


def shuffle_grid_patches(image_array, grid_size, permutation):
    height, width, channels = image_array.shape
    patch_height = height // grid_size
    patch_width = width // grid_size
    shuffled = np.zeros_like(image_array)

    patches = []
    for row in range(grid_size):
        for col in range(grid_size):
            y0 = row * patch_height
            y1 = (row + 1) * patch_height
            x0 = col * patch_width
            x1 = (col + 1) * patch_width
            patches.append(image_array[y0:y1, x0:x1, :])

    for target_index, source_index in enumerate(permutation):
        row = target_index // grid_size
        col = target_index % grid_size
        y0 = row * patch_height
        y1 = (row + 1) * patch_height
        x0 = col * patch_width
        x1 = (col + 1) * patch_width
        shuffled[y0:y1, x0:x1, :] = patches[source_index]

    return shuffled


def apply_challenge_variant(image, variant, sample_seed):
    if variant == 'clean':
        return image

    rng = np.random.default_rng(sample_seed)
    image_array = np.array(image, dtype=np.uint8)

    if variant == 'texture':
        permutation = rng.permutation(16)
        shuffled = shuffle_grid_patches(image_array, grid_size=4, permutation=permutation)
        mixed = np.clip(0.70 * shuffled + 0.30 * image_array, 0, 255).astype(np.uint8)
        return Image.fromarray(mixed)

    if variant == 'long_range':
        coarse = image.resize((8, 8), Image.Resampling.BILINEAR).resize(image.size, Image.Resampling.NEAREST)
        coarse_array = np.array(coarse, dtype=np.uint8)
        return Image.fromarray(coarse_array)

    if variant == 'damaged_occluded':
        damaged = image_array.copy()
        height, width, _ = damaged.shape

        occlusion_width = max(6, width // 3)
        occlusion_height = max(6, height // 3)
        x0 = int(rng.integers(width // 6, max(width // 6 + 1, width - occlusion_width)))
        y0 = int(rng.integers(height // 6, max(height // 6 + 1, height - occlusion_height)))
        damaged[y0:y0 + occlusion_height, x0:x0 + occlusion_width, :] = 0

        for _ in range(2):
            stripe_y = int(rng.integers(0, max(1, height - 3)))
            stripe_thickness = int(rng.integers(2, 4))
            damaged[stripe_y:stripe_y + stripe_thickness, :, :] = 255

        for _ in range(10):
            px = int(rng.integers(0, width))
            py = int(rng.integers(0, height))
            damaged[py, px, :] = np.array([255, 255, 255], dtype=np.uint8)

        return Image.fromarray(damaged)

    raise ValueError(f'Unsupported challenge variant: {variant}')


class CIFAR100SmallChallengeDataset(Dataset):
    def __init__(
        self,
        root,
        train,
        transform,
        selected_classes,
        samples_per_class,
        challenge_variant='clean',
        seed=42,
        download=True,
    ):
        self.base_dataset = datasets.CIFAR100(root=str(root), train=train, download=download)
        self.transform = transform
        self.challenge_variant = challenge_variant
        self.classes = list(selected_classes)
        self.class_to_idx = {class_name: class_index for class_index, class_name in enumerate(self.classes)}
        self.original_class_to_idx = self.base_dataset.class_to_idx
        self.original_labels = [self.original_class_to_idx[class_name] for class_name in self.classes]
        self.new_label_by_original = {original_label: new_label for new_label, original_label in enumerate(self.original_labels)}
        self.seed = seed + (0 if train else 10_000)

        rng = random.Random(self.seed)
        subset_indices = build_balanced_subset_indices(
            self.base_dataset.targets,
            self.original_labels,
            samples_per_class,
            rng,
        )

        self.samples = [
            (sample_index, self.new_label_by_original[self.base_dataset.targets[sample_index]])
            for sample_index in subset_indices
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample_index, label = self.samples[index]
        image = Image.fromarray(self.base_dataset.data[sample_index]).convert('RGB')
        image = apply_challenge_variant(
            image,
            self.challenge_variant,
            sample_seed=self.seed + sample_index * 17 + index,
        )

        if self.transform is not None:
            image = self.transform(image)

        return image, label


In [5]:
def create_variant_datasets(
    variant,
    alexnet_train_transform,
    alexnet_eval_transform,
    dhvt_train_transform,
    dhvt_eval_transform,
):
    alexnet_trainset = CIFAR100SmallChallengeDataset(
        root=data_dir,
        train=True,
        transform=alexnet_train_transform,
        selected_classes=selected_cifar100_classes,
        samples_per_class=train_samples_per_class,
        challenge_variant=variant,
        seed=seed,
    )
    alexnet_valset = CIFAR100SmallChallengeDataset(
        root=data_dir,
        train=False,
        transform=alexnet_eval_transform,
        selected_classes=selected_cifar100_classes,
        samples_per_class=val_samples_per_class,
        challenge_variant=variant,
        seed=seed,
    )
    dhvt_trainset = CIFAR100SmallChallengeDataset(
        root=data_dir,
        train=True,
        transform=dhvt_train_transform,
        selected_classes=selected_cifar100_classes,
        samples_per_class=train_samples_per_class,
        challenge_variant=variant,
        seed=seed,
    )
    dhvt_valset = CIFAR100SmallChallengeDataset(
        root=data_dir,
        train=False,
        transform=dhvt_eval_transform,
        selected_classes=selected_cifar100_classes,
        samples_per_class=val_samples_per_class,
        challenge_variant=variant,
        seed=seed,
    )
    return alexnet_trainset, alexnet_valset, dhvt_trainset, dhvt_valset


classes = selected_cifar100_classes
num_classes = len(classes)
print('Number of classes:', num_classes)


Number of classes: 12


In [6]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super(AlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


In [7]:
class SOPE(nn.Module):
    def __init__(self, in_ch=3, embed_dim=192, patch_size=4):
        super().__init__()
        if patch_size < 1 or (patch_size & (patch_size - 1)) != 0:
            raise ValueError("patch_size must be a power of 2")

        num_stages = int(math.log2(patch_size))
        if num_stages == 0:
            raise ValueError("patch_size must be at least 2")

        self.pre_scale = nn.Parameter(torch.ones(1, in_ch, 1, 1))
        self.pre_bias = nn.Parameter(torch.zeros(1, in_ch, 1, 1))
        self.post_scale = nn.Parameter(torch.ones(1, embed_dim, 1, 1))
        self.post_bias = nn.Parameter(torch.zeros(1, embed_dim, 1, 1))

        layers = []
        in_dim = in_ch
        for _ in range(num_stages):
            layers.extend([
                nn.Conv2d(in_dim, embed_dim, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(embed_dim),
                nn.GELU(),
            ])
            in_dim = embed_dim
        self.conv_layers = nn.Sequential(*layers)

    def forward(self, x):
        x = x * self.pre_scale + self.pre_bias
        x = self.conv_layers(x)
        x = x * self.post_scale + self.post_bias
        B, C, H, W = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        return tokens, H, W

class DAFF(nn.Module):
    def __init__(self, dim):
        super().__init__()
        hidden_dim = dim * 4
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.dwconv = nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, padding=1, groups=hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.channel_fc = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, dim),
            nn.Sigmoid()
        )

    def forward(self, x, H, W):
        cls_token = x[:, :1]
        patch_tokens = x[:, 1:]
        B, N, C = patch_tokens.shape
        assert H * W == N

        patch_tokens = self.fc1(patch_tokens)
        shortcut = patch_tokens

        patch_tokens = patch_tokens.transpose(1, 2).reshape(B, C * 4, H, W)
        patch_tokens = self.dwconv(patch_tokens)
        patch_tokens = patch_tokens.reshape(B, C * 4, N).transpose(1, 2)
        patch_tokens = self.act(patch_tokens)
        patch_tokens = patch_tokens + shortcut
        patch_tokens = self.fc2(patch_tokens)

        channel_weight = patch_tokens.mean(dim=1)
        channel_weight = self.channel_fc(channel_weight)
        cls_token = cls_token * channel_weight.unsqueeze(1)

        return torch.cat([cls_token, patch_tokens], dim=1)

class HeadTokenAttention(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        if dim % num_heads != 0:
            raise ValueError("dim must be divisible by num_heads")

        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.head_proj = nn.Linear(self.head_dim, dim)
        self.head_act = nn.GELU()
        self.head_embedding = nn.Parameter(torch.zeros(1, num_heads, dim))

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.last_attn = None

    def forward(self, x):
        B, N, C = x.shape

        grouped_tokens = x.reshape(B, N, self.num_heads, self.head_dim)
        head_tokens = grouped_tokens.mean(dim=1)
        head_tokens = self.head_proj(head_tokens)
        head_tokens = self.head_act(head_tokens)
        head_tokens = head_tokens + self.head_embedding

        x_with_heads = torch.cat([x, head_tokens], dim=1)
        total_tokens = x_with_heads.shape[1]

        qkv = self.qkv(x_with_heads).reshape(
            B, total_tokens, 3, self.num_heads, self.head_dim
        )
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        self.last_attn = attn.detach()

        out = attn @ v
        out = out.transpose(1, 2).reshape(B, total_tokens, C)
        out = self.proj(out)

        updated_cls = out[:, :1] + out[:, N:].mean(dim=1, keepdim=True)
        updated_patch_tokens = out[:, 1:N]
        return torch.cat([updated_cls, updated_patch_tokens], dim=1)

class DHVTBlock(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = HeadTokenAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = DAFF(dim)

    def forward(self, x, H, W):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x), H, W)
        return x

class DHVT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, embed_dim=192, depth=12, num_heads=4, num_classes=10):
        super().__init__()
        self.patch_embed = SOPE(3, embed_dim, patch_size=patch_size)
        num_patches = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))
        self.blocks = nn.ModuleList([DHVTBlock(embed_dim, num_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x, H, W = self.patch_embed(x)
        B = x.shape[0]
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls, x), dim=1)
        x = x + self.pos_embed
        for blk in self.blocks:
            x = blk(x, H, W)
        x = self.norm(x)
        cls = x[:, 0]
        return self.head(cls)


In [8]:
def create_alexnet():
    model = AlexNet(num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=alexnet_learning_rate, weight_decay=weight_decay)
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    return model, criterion, optimizer, scheduler


def create_dhvt():
    model = DHVT(img_size=32, embed_dim=192, depth=12, num_heads=4, num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=dhvt_learning_rate, weight_decay=weight_decay)
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    return model, criterion, optimizer, scheduler


def create_loader(dataset, batch_size, shuffle):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


def train_model(model_name, variant, model, criterion, optimizer, scheduler, trainloader, valloader):
    history = []
    best_val_accuracy = 0.0
    best_model_path = model_dir / f'{dataset_name}_{variant}_{model_name}_best.pth'
    last_model_path = model_dir / f'{dataset_name}_{variant}_{model_name}_last.pth'
    train_start_time = time.time()

    print(f'\nStart training {model_name} on {variant}...', flush=True)
    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        model.train()
        running_loss = 0.0
        running_correct = 0
        total_train = 0

        for inputs, labels in trainloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)
            running_correct += (predictions == labels).sum().item()
            total_train += labels.size(0)

        model.eval()
        val_loss = 0.0
        val_correct = 0
        total_val = 0

        with torch.inference_mode():
            for inputs, labels in valloader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * labels.size(0)
                predictions = outputs.argmax(dim=1)
                val_correct += (predictions == labels).sum().item()
                total_val += labels.size(0)

        learning_rate = optimizer.param_groups[0]['lr']
        scheduler.step()

        train_loss = running_loss / total_train
        train_accuracy = running_correct / total_train
        validation_loss = val_loss / total_val
        validation_accuracy = val_correct / total_val
        epoch_time = time.time() - epoch_start_time

        history.append({
            'epoch': epoch + 1,
            'variant': variant,
            'model': model_name,
            'train_loss': train_loss,
            'train_accuracy': train_accuracy,
            'validation_loss': validation_loss,
            'validation_accuracy': validation_accuracy,
            'learning_rate': learning_rate,
            'epoch_time_seconds': epoch_time,
        })

        print(
            f"{model_name} | {variant} | epoch {epoch + 1}/{num_epochs} | "
            f"train acc: {train_accuracy:.4f} | val acc: {validation_accuracy:.4f} | "
            f"train loss: {train_loss:.4f} | val loss: {validation_loss:.4f} | "
            f"lr: {learning_rate:.6f} | time: {epoch_time:.2f}s",
            flush=True
        )

        checkpoint = {
            'epoch': epoch + 1,
            'variant': variant,
            'model_name': model_name,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history': history,
            'best_val_accuracy': best_val_accuracy,
            'class_names': classes,
            'seed': seed,
        }
        torch.save(checkpoint, last_model_path)

        if validation_accuracy > best_val_accuracy:
            best_val_accuracy = validation_accuracy
            checkpoint['best_val_accuracy'] = best_val_accuracy
            torch.save(checkpoint, best_model_path)

    total_time = time.time() - train_start_time
    best_epoch = max(history, key=lambda item: item['validation_accuracy'])
    return {
        'model_name': model_name,
        'variant': variant,
        'history': history,
        'best_val_accuracy': best_val_accuracy,
        'best_epoch': best_epoch['epoch'],
        'final_val_accuracy': history[-1]['validation_accuracy'],
        'training_time_seconds': total_time,
        'parameter_count': count_parameters(model),
        'best_model_path': str(best_model_path),
    }


def render_summary_table(rows, columns, title, figure_path):
    table_text = []
    for row in rows:
        table_text.append([str(row[column]) for column in columns])

    figure_height = max(2.5, 0.55 * len(rows) + 1.3)
    plt.figure(figsize=(max(10, 2.0 * len(columns)), figure_height))
    plt.axis('off')
    table = plt.table(
        cellText=table_text,
        colLabels=columns,
        cellLoc='center',
        loc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.4)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(figure_path, dpi=300, bbox_inches='tight')
    plt.show()


def save_csv(rows, columns, path):
    with path.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=columns)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


In [9]:
alexnet_train_transform = transforms.Compose([
    transforms.Resize((cnn_image_size, cnn_image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

alexnet_eval_transform = transforms.Compose([
    transforms.Resize((cnn_image_size, cnn_image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dhvt_train_transform = transforms.Compose([
    transforms.Resize((dhvt_image_size, dhvt_image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(dhvt_image_size, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dhvt_eval_transform = transforms.Compose([
    transforms.Resize((dhvt_image_size, dhvt_image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

experiment_results = []
history_rows = []

for variant in experiment_variants:
    print('\n' + '=' * 80)
    print(f'Variant: {variant} | {challenge_variant_descriptions[variant]}')
    print('=' * 80)

    alex_trainset, alex_valset, dhvt_trainset, dhvt_valset = create_variant_datasets(
        variant,
        alexnet_train_transform,
        alexnet_eval_transform,
        dhvt_train_transform,
        dhvt_eval_transform,
    )

    alexnet_trainloader = create_loader(alex_trainset, cnn_batch_size, shuffle=True)
    alexnet_valloader = create_loader(alex_valset, cnn_batch_size, shuffle=False)
    dhvt_trainloader = create_loader(dhvt_trainset, dhvt_batch_size, shuffle=True)
    dhvt_valloader = create_loader(dhvt_valset, dhvt_batch_size, shuffle=False)

    alexnet_model, alexnet_criterion, alexnet_optimizer, alexnet_scheduler = create_alexnet()
    alexnet_result = train_model(
        'AlexNet',
        variant,
        alexnet_model,
        alexnet_criterion,
        alexnet_optimizer,
        alexnet_scheduler,
        alexnet_trainloader,
        alexnet_valloader,
    )

    dhvt_model, dhvt_criterion, dhvt_optimizer, dhvt_scheduler = create_dhvt()
    dhvt_result = train_model(
        'DHVT',
        variant,
        dhvt_model,
        dhvt_criterion,
        dhvt_optimizer,
        dhvt_scheduler,
        dhvt_trainloader,
        dhvt_valloader,
    )

    experiment_results.append({
        'variant': variant,
        'description': challenge_variant_descriptions[variant],
        'alexnet_best_val_accuracy': round(alexnet_result['best_val_accuracy'], 4),
        'dhvt_best_val_accuracy': round(dhvt_result['best_val_accuracy'], 4),
        'gap_dhvt_minus_alexnet': round(dhvt_result['best_val_accuracy'] - alexnet_result['best_val_accuracy'], 4),
        'alexnet_best_epoch': alexnet_result['best_epoch'],
        'dhvt_best_epoch': dhvt_result['best_epoch'],
        'alexnet_time_seconds': round(alexnet_result['training_time_seconds'], 2),
        'dhvt_time_seconds': round(dhvt_result['training_time_seconds'], 2),
        'alexnet_params': alexnet_result['parameter_count'],
        'dhvt_params': dhvt_result['parameter_count'],
    })

    history_rows.extend(alexnet_result['history'])
    history_rows.extend(dhvt_result['history'])



Variant: clean | baseline balanced subset without synthetic corruption


100%|██████████| 169M/169M [00:03<00:00, 43.3MB/s] 



Start training AlexNet on clean...
AlexNet | clean | epoch 1/8 | train acc: 0.0771 | val acc: 0.1167 | train loss: 2.4930 | val loss: 2.3096 | lr: 0.001000 | time: 5.92s
AlexNet | clean | epoch 2/8 | train acc: 0.1292 | val acc: 0.1583 | train loss: 2.2907 | val loss: 2.2601 | lr: 0.000962 | time: 4.62s
AlexNet | clean | epoch 3/8 | train acc: 0.1625 | val acc: 0.1833 | train loss: 2.1596 | val loss: 2.1158 | lr: 0.000854 | time: 5.06s
AlexNet | clean | epoch 4/8 | train acc: 0.1823 | val acc: 0.2167 | train loss: 2.0685 | val loss: 2.0904 | lr: 0.000691 | time: 4.86s
AlexNet | clean | epoch 5/8 | train acc: 0.2292 | val acc: 0.2556 | train loss: 1.9916 | val loss: 2.0054 | lr: 0.000500 | time: 5.39s
AlexNet | clean | epoch 6/8 | train acc: 0.2656 | val acc: 0.2889 | train loss: 1.9149 | val loss: 1.9841 | lr: 0.000309 | time: 5.57s
AlexNet | clean | epoch 7/8 | train acc: 0.2646 | val acc: 0.2972 | train loss: 1.8549 | val loss: 1.9691 | lr: 0.000146 | time: 4.79s
AlexNet | clean | e

In [12]:
experiment_results_sorted = sorted(
    experiment_results,
    key=lambda item: item['gap_dhvt_minus_alexnet'],
    reverse=True,
)

print('Benchmark summary:')
for row in experiment_results_sorted:
    print(
        f"{row['variant']}: AlexNet={row['alexnet_best_val_accuracy']:.4f} | "
        f"DHVT={row['dhvt_best_val_accuracy']:.4f} | "
        f"gap={row['gap_dhvt_minus_alexnet']:+.4f}"
    )

summary_csv_path = prediction_dir / f'{dataset_name}_multivariant_summary.csv'
history_csv_path = prediction_dir / f'{dataset_name}_multivariant_histories.csv'

save_csv(experiment_results_sorted, list(experiment_results_sorted[0].keys()), summary_csv_path)
save_csv(history_rows, list(history_rows[0].keys()), history_csv_path)

print('Summary CSV:', summary_csv_path)
print('History CSV:', history_csv_path)


Benchmark summary:
clean: AlexNet=0.2694 | DHVT=0.5167 | gap=+0.2472
texture: AlexNet=0.2583 | DHVT=0.4278 | gap=+0.1694
damaged_occluded: AlexNet=0.2944 | DHVT=0.4222 | gap=+0.1278
long_range: AlexNet=0.3333 | DHVT=0.4472 | gap=+0.1139
Summary CSV: /kaggle/working/predictions/cifar100_v4_small_multivariant_summary.csv
History CSV: /kaggle/working/predictions/cifar100_v4_small_multivariant_histories.csv
